Source: @DeepCharts Youtube Channel (https://www.youtube.com/@DeepCharts)

# Refactored: Async Multi-Symbol Technical Indicators for Machine Learning

This refactor adapts the original notebook to process many symbols using asynchronous downloads and a lightweight Cython-ready acceleration path for a few numeric feature helpers. It also reads symbols from a CSV already present in the repo tree.

## 1. Imports

In [ ]:
import asyncio
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

try:
    import pyximport
    pyximport.install(language_level=3, inplace=False)
    CYTHON_AVAILABLE = True
except Exception:
    CYTHON_AVAILABLE = False

MAX_WORKERS = 24


## 2. Symbol source CSV

In [ ]:
SYMBOL_CSV = Path('../csv/INTC_1d_1y.csv')

def load_symbols(symbol_csv: Path, fallback=None):
    fallback = fallback or ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD']
    if not symbol_csv.exists():
        return fallback

    stem = symbol_csv.stem
    primary = stem.split('_')[0].strip().upper()
    symbols = [primary] if primary else []

    try:
        raw = pd.read_csv(symbol_csv)
        for col in raw.columns:
            if str(col).lower() in {'symbol', 'symbols', 'ticker', 'tickers'}:
                vals = raw[col].dropna().astype(str).str.strip().str.upper().tolist()
                symbols.extend(vals)
    except Exception:
        pass

    symbols = [s for s in dict.fromkeys(symbols) if s]
    return symbols or fallback

symbols = load_symbols(SYMBOL_CSV)
symbols[:10], len(symbols)

## 3. Optional Cython helper generation

In [ ]:
CYTHON_MODULE = Path('finance_cython_helpers.pyx')
CYTHON_SOURCE = '''
import numpy as np
cimport numpy as np

def pct_change_arr(np.ndarray[np.float64_t, ndim=1] arr):
    cdef Py_ssize_t i, n = arr.shape[0]
    cdef np.ndarray[np.float64_t, ndim=1] out = np.empty(n, dtype=np.float64)
    out[0] = np.nan
    for i in range(1, n):
        if arr[i-1] == 0:
            out[i] = np.nan
        else:
            out[i] = (arr[i] - arr[i-1]) / arr[i-1]
    return out

def intraday_range(np.ndarray[np.float64_t, ndim=1] high, np.ndarray[np.float64_t, ndim=1] low):
    cdef Py_ssize_t i, n = high.shape[0]
    cdef np.ndarray[np.float64_t, ndim=1] out = np.empty(n, dtype=np.float64)
    for i in range(n):
        out[i] = high[i] - low[i]
    return out
'''

if CYTHON_AVAILABLE and not CYTHON_MODULE.exists():
    CYTHON_MODULE.write_text(CYTHON_SOURCE)

if CYTHON_AVAILABLE:
    try:
        import finance_cython_helpers as fch
    except Exception:
        fch = None
else:
    fch = None

fch is not None

## 4. Async download pipeline

In [ ]:
def download_symbol(symbol, start='2022-10-25', end='2024-10-25', interval='1d'):
    df = yf.download(symbol, start=start, end=end, interval=interval, progress=False, auto_adjust=False, threads=False)
    if df is None or df.empty:
        return symbol, pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    keep = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in df.columns]
    return symbol, df[keep].copy()

async def download_symbol_async(symbol, executor):
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(executor, download_symbol, symbol)

async def download_many(symbols, max_workers=MAX_WORKERS):
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tasks = [download_symbol_async(symbol, executor) for symbol in symbols]
        for symbol, df in await asyncio.gather(*tasks):
            if df is not None and not df.empty:
                results[symbol] = df
    return results

price_map = await download_many(symbols)
list(price_map.keys())[:10], len(price_map)

## 5. Feature engineering

In [ ]:
def add_fast_numeric_features(df):
    close_vals = df['Close_shifted'].astype(float).to_numpy()
    high_vals = df['High_shifted'].astype(float).to_numpy()
    low_vals = df['Low_shifted'].astype(float).to_numpy()

    if fch is not None:
        df['Pct_Change_1'] = fch.pct_change_arr(close_vals)
        df['Intraday_Range'] = fch.intraday_range(high_vals, low_vals)
    else:
        df['Pct_Change_1'] = df['Close_shifted'].pct_change()
        df['Intraday_Range'] = df['High_shifted'] - df['Low_shifted']

    return df

def engineer_features(df, symbol):
    df = df.copy()
    df['Symbol'] = symbol

    df['Previous_Close'] = df['Close'].shift(1)
    df['Close_shifted'] = df['Close'].shift(1)
    df['Open_shifted'] = df['Open'].shift(1)
    df['High_shifted'] = df['High'].shift(1)
    df['Low_shifted'] = df['Low'].shift(1)

    df['SMA_50'] = ta.sma(df['Close_shifted'], length=50)
    df['EMA_50'] = ta.ema(df['Close_shifted'], length=50)
    df['RSI'] = ta.rsi(df['Close_shifted'], length=14)

    macd = ta.macd(df['Close_shifted'], fast=12, slow=26, signal=9)
    df['MACD'] = macd['MACD_12_26_9']
    df['Signal_Line'] = macd['MACDs_12_26_9']

    bollinger = ta.bbands(df['Close_shifted'], length=20, std=2)
    df['BB_Upper'] = bollinger['BBU_20_2.0']
    df['BB_Middle'] = bollinger['BBM_20_2.0']
    df['BB_Lower'] = bollinger['BBL_20_2.0']

    stoch = ta.stoch(df['High_shifted'], df['Low_shifted'], df['Close_shifted'], k=14, d=3)
    df['%K'] = stoch['STOCHk_14_3_3']
    df['%D'] = stoch['STOCHd_14_3_3']

    df['ATR'] = ta.atr(df['High_shifted'], df['Low_shifted'], df['Close_shifted'], length=14)
    df = add_fast_numeric_features(df)
    df.dropna(inplace=True)
    return df

feature_frames = [engineer_features(df, symbol) for symbol, df in price_map.items() if not df.empty]
dataset = pd.concat(feature_frames).reset_index().rename(columns={'index': 'Date'}) if feature_frames else pd.DataFrame()
dataset.head()

## 6. Quick regression example

In [ ]:
model_df = dataset.copy()
model_df['Target'] = model_df.groupby('Symbol')['Close'].shift(-1)
model_df.dropna(inplace=True)

feature_cols = [
    'Previous_Close', 'SMA_50', 'EMA_50', 'RSI', 'MACD', 'Signal_Line',
    'BB_Upper', 'BB_Middle', 'BB_Lower', '%K', '%D', 'ATR',
    'Pct_Change_1', 'Intraday_Range'
]

train_size = int(len(model_df) * 0.8)
train_df = model_df.iloc[:train_size].copy()
test_df = model_df.iloc[train_size:].copy()

X_train = sm.add_constant(train_df[feature_cols])
y_train = train_df['Target']
X_test = sm.add_constant(test_df[feature_cols], has_constant='add')
y_test = test_df['Target']

model = sm.OLS(y_train, X_train).fit()
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
pd.DataFrame({'MAE': [mae], 'RMSE': [rmse], 'Rows': [len(model_df)], 'Symbols': [model_df['Symbol'].nunique()]})